
### Что есть в блокноте сейчас

Уравнение ротора H в блокноте записано в ячейках 49:

```python
eq_rot_H_left_conductor_y = rot_H_left_conductor_y == -(I*omega/c * D_l[1])[0]
```

где `D_l = epsilon' * E` — вектор электрического смещения. Это **однородное** уравнение Максвелла без какого-либо стороннего тока. Правая часть содержит только `ε'E` — поляризационный ток проводящей среды.

Граничные условия в `sys4`  — четыре уравнения непрерывности полей на границах `x = ±a`:

```python
eqHz_l:  H_left_conductor_sol_z(-a) == H_inner_vacuum_sol_z(-a)   # непрерывность Hz
eq_Ey_l: E_left_conductor_sub_y(-a) == E_inner_vacuum_sub_y(-a)   # непрерывность Ey
```

Обе эти строки — условия без скачка. Правая часть каждого уравнения содержит поле вакуума, левая — поле металла. **Никакого стороннего тока нигде нет.**

---



### Что нужно сделать — два места в блокноте

Для корректного введения impressed current нужно изменить **ровно два места**:

**Место 1 — уравнение ротора H в металле** (ячейка ~344). Сейчас:
```python
eq_rot_H_left_conductor_y = rot_H_left_conductor_y == -(I*omega/c * D_l[1])[0]
```
Должно стать:
```python
eq_rot_H_left_conductor_y = rot_H_left_conductor_y == -(I*omega/c * D_l[1])[0] + 4*pi/c * j_ext_y(x)
```

**Место 2 — граничное условие непрерывности Hz** (ячейка 418). Сейчас:
```python
eqHz_l = H_left_conductor_sol_z(-a) == H_inner_vacuum_sol_z(-a)
```
Это условие выводится из интегральной формы ротора H. Когда в объёме металла есть `j_ext`, скачка Hz на границе нет — но только если `j_ext` — объёмный ток. Если же вы хотите **поверхностный** ток `J_s` (дельта-функция на границе), то возникает скачок:
```python
eqHz_l = H_inner_vacuum_sol_z(-a) - H_left_conductor_sol_z(-a) == 4*pi/c * J_s_ext
```

---

### Какой вариант физически правильнее для вашей задачи

Объёмный скин-ток (то, что реализовано в файле из прошлой ячейки) — физически правильнее: ток реально течёт по глубине скин-слоя, и его профиль задаётся как `j_ext(x) = j0 * exp(gamma*(x+a))`. В этом случае граничное условие Hz **не меняется**, а `j_ext` входит только в правую часть волнового уравнения, что и порождает частное решение `P_d`.

Поверхностный ток — предельный случай при `delta_s → 0`. Он проще, но менее точен при конечной проводимости.

---

### Конкретный патч для ячейки 49

Вот точная замена, которую нужно вставить вместо ячейки 49:Вот итоговый ответ по существу вопроса.

---

## Что реализовано, а что нет

**Не реализовано ни одно из двух.** В блокноте уравнение ротора H записано в ячейках 343–347 в однородном виде:

```python
eq_rot_H_left_conductor_y = rot_H_left_conductor_y == -(I*omega/c * D_l[1])[0]
```

Никакого `j_ext` в правой части нет. Граничные условия в `sys4` (ячейки 415–419) — стандартные условия непрерывности полей без скачка. Матрица `M4_K` строится из этой однородной системы и решается как задача на собственные значения (`det = 0`).

---

## Что делают два файла

| Файл | Уровень | Что делает |
|---|---|---|
| `j_ext_impressed_current.py` | Численный | Строит частное решение `P_d` аналитически, не трогая `M4_K`. Считает 4 члена силы числом. |
| `patch_rot_H.py` | Символьный | Модифицирует `eq_rot_H_left_conductor_y`, пересчитывает `E_y_l` через `get_field()`, строит `sys4_ext` и разделяет `M4_K_ext` и `B_ext` символьно. |

Физика одинакова — это два пути к одному результату. Разница практическая: символьный путь (`patch_rot_H.py`) позволяет увидеть, как `j_ext` меняет аналитическое выражение для `E_y_l(x)` до подстановки чисел, и работает с вашей существующей функцией `get_field()`. Численный путь (`j_ext_impressed_current.py`) быстрее запускается и сразу выдаёт силу.

Рекомендую начать с численного — проверить, что `F1 ≈ calc_p_Ht_left_jeH_conj` при `j0_norm_d = 0`, а потом при необходимости добавить символьный патч.